In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv("caesarian.csv")
print(df.sample(5))

    Age  DeliveryNumber  DeliveryTime  BloodPressure  HeartProblem  Caesarian
0    22               1             0              2             0          0
77   29               2             1              2             0          1
32   32               2             0              2             1          1
58   26               1             0              2             0          1
38   31               1             0              1             0          0


In [31]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [32]:
X.shape, y.shape

((80, 5), (80,))

In [35]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [38]:
# 2. Split Data (for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Create a Pipeline
# Scaling is strictly required for Logistic Regression to converge correctly
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42))
])

# 4. Define Hyperparameter Grid
param_grid = [
    # Solver 'liblinear' works well for small datasets and supports both L1 and L2
    {
        'logreg__solver': ['liblinear'],
        'logreg__penalty': ['l1', 'l2'],
        'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    # Solver 'lbfgs' is the default and supports L2
    {
        'logreg__solver': ['lbfgs'],
        'logreg__penalty': ['l2'],
        'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100]
    }
]

# 5. Run Grid Search with Cross-Validation
# StratifiedKFold is safer for small datasets to preserve class balance
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)

grid_search.fit(X_train, y_train)

# 6. Results
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")
print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Best Parameters: {'logreg__C': 1, 'logreg__penalty': 'l2', 'logreg__solver': 'liblinear'}
Best Cross-Validation Accuracy: 0.6577
Test Set Accuracy: 0.6250
